![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 05: NLP for Clinical Text
***
**Learning objectives**
- Build BoW and TF-IDF representations of clinical notes
- Apply clinical-domain stop words and n-gram features
- Train classifiers for condition detection and readmission prediction from notes
- Compare BoW/TF-IDF vs character n-grams for clinical text
- Visualise discriminative terms with log-odds analysis

**Estimated time:** 2.5 hours | **Level:** Intermediate | **Libraries:** `sklearn`, `pandas`, `numpy`


## 1. Setup & Synthetic Clinical Corpus

In [ ]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score,
                              average_precision_score, f1_score, confusion_matrix)
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)

CONDITIONS = {
    "Cardiac"  : [
        "heart failure BNP elevated furosemide diuresis ejection fraction reduced cardiomegaly edema IV",
        "STEMI myocardial infarction troponin elevated ST elevation cath lab PCI stent aspirin heparin",
        "atrial fibrillation rate control metoprolol digoxin anticoagulation warfarin apixaban stroke",
        "hypertensive urgency BP elevated labetalol nicardipine end organ damage creatinine troponin",
    ],
    "Pulmonary": [
        "COPD exacerbation bronchospasm albuterol ipratropium methylprednisolone azithromycin oxygen SpO2",
        "community acquired pneumonia consolidation fever WBC antibiotics ceftriaxone blood cultures",
        "pulmonary embolism bilateral CT angiography rivaroxaban heparin DVT RV strain troponin",
        "asthma exacerbation peak flow bronchodilators steroids magnesium sulfate admission ICU",
    ],
    "Diabetes" : [
        "diabetic ketoacidosis glucose anion gap metabolic acidosis insulin drip IV fluids potassium",
        "hypoglycemia glucose altered mental status dextrose glucagon diabetes education carbohydrate",
        "uncontrolled type 2 diabetes HbA1c polyuria polydipsia weight loss metformin GLP-1 SGLT2",
        "diabetic foot infection wound culture debridement surgical amputation vascular IV antibiotics",
    ],
    "Renal"    : [
        "acute kidney injury creatinine elevated fluid resuscitation hold nephrotoxins ACE dialysis",
        "end stage renal disease hemodialysis access fistula potassium hyperkalemia emergent dialysis",
        "nephrotic syndrome proteinuria albumin low edema diuretics renal biopsy FSGS prednisone",
    ],
    "Sepsis"   : [
        "septic shock norepinephrine vasopressor broad spectrum antibiotics vancomycin meropenem lactate ICU",
        "bacteremia Staph aureus MSSA blood cultures positive oxacillin nafcillin endocarditis vegetation",
        "neutropenic fever chemotherapy immunosuppressed filgrastim antifungal fluconazole cultures negative",
    ],
}

def augment(text, noise=0.15):
    words = text.split()
    result = []
    for w in words:
        result.append(w)
        if np.random.rand() < noise:
            result.append(np.random.choice(["assessment","plan","patient","stable","monitor","admitted"]))
    return " ".join(result)

corpus = []
for condition, templates in CONDITIONS.items():
    readmit_prob = {"Cardiac":0.26,"Pulmonary":0.18,"Diabetes":0.21,"Renal":0.23,"Sepsis":0.31}
    for i in range(100):
        base = np.random.choice(templates)
        text = augment(base)
        readmitted = int(np.random.rand() < readmit_prob[condition])
        corpus.append({"text":text,"condition":condition,"readmitted":readmitted,
                       "note_length":len(text.split())})

df = pd.DataFrame(corpus)
print(f"Corpus: {len(df)} notes | Readmission rate: {df.readmitted.mean()*100:.1f}%")
print(df.condition.value_counts())


## 2. Clinical Stop Words & Text Cleaning

In [ ]:
CLINICAL_STOP_WORDS = [
    "patient","pt","the","a","an","is","was","were","been","has","have","had",
    "will","would","could","should","may","might","must","can","do","does","did",
    "and","or","but","with","for","on","at","in","to","of","from","by","as","be",
    "this","that","these","those","it","its","all","any","both","each","few",
    "most","other","some","no","not","only","same","too","very","just",
    "admitted","admission","discharge","plan","assessment","note","hospital",
    "day","follow","given","started","continued","current","status","history",
    "exam","also","during","after","before","within","without","between",
]

def clean_clinical_text(text):
    text = text.lower()
    text = re.sub(r'\b\d+\.?\d*\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_clinical_text)
print("Sample cleaned text:")
print(df["clean_text"].iloc[0][:200])


## 3. Feature Extraction Comparison

In [ ]:
X = df["clean_text"]; y_cond = df["condition"]; y_re = df["readmitted"]
X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
    X, y_cond, y_re, test_size=0.20, stratify=y_cond, random_state=42)

feature_configs = {
    "BoW (unigrams)"     : CountVectorizer(stop_words=CLINICAL_STOP_WORDS, max_features=3000),
    "TF-IDF (unigrams)"  : TfidfVectorizer(stop_words=CLINICAL_STOP_WORDS, max_features=3000),
    "TF-IDF (1-2 grams)" : TfidfVectorizer(stop_words=CLINICAL_STOP_WORDS, ngram_range=(1,2), max_features=5000),
    "TF-IDF (char 3-5)"  : TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=5000),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
print("Feature comparison — 5-fold CV:\n")
feat_results = []
for name, vect in feature_configs.items():
    pipe = Pipeline([("v",vect),("c",LogisticRegression(C=1,max_iter=1000,random_state=42,multi_class="multinomial"))])
    f1_cond = cross_val_score(pipe, X, y_cond, cv=skf, scoring="f1_weighted").mean()
    auc_re   = cross_val_score(pipe, X, y_re,   cv=skf, scoring="roc_auc").mean()
    feat_results.append({"Feature":name,"Condition F1":round(f1_cond,4),"Readmit AUC":round(auc_re,4)})
    print(f"  {name:28s}: Condition F1={f1_cond:.4f} | Readmit AUC={auc_re:.4f}")
display(pd.DataFrame(feat_results))


## 4. Condition Classification — Full Evaluation

In [ ]:
vect = TfidfVectorizer(stop_words=CLINICAL_STOP_WORDS, ngram_range=(1,2),
                        max_features=5000, sublinear_tf=True)
clf  = LogisticRegression(C=1, max_iter=1000, random_state=42, multi_class="multinomial")
pipe = Pipeline([("v",vect),("c",clf)])
pipe.fit(X_tr, yc_tr)
yc_pred = pipe.predict(X_te)

print("Condition Classification Report:")
print(classification_report(yc_te, yc_pred))

classes = sorted(y_cond.unique())
cm = confusion_matrix(yc_te, yc_pred, labels=classes)
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix — Condition Classification")

fn = pipe.named_steps["v"].get_feature_names_out()
coefs = pipe.named_steps["c"].coef_
for i, cond in enumerate(pipe.named_steps["c"].classes_):
    top_idx = np.argsort(coefs[i])[-8:]
    axes[1].barh([fn[j] for j in top_idx][::-1], [coefs[i][j] for j in top_idx][::-1],
                 alpha=0.6, label=cond)
axes[1].set_xlabel("Log-odds coefficient")
axes[1].set_title("Top TF-IDF features by condition")
axes[1].legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig("/tmp/mod05/tfidf_classification.png", bbox_inches="tight"); plt.show()


## 5. Readmission Prediction from Clinical Notes

In [ ]:
vect_re = TfidfVectorizer(stop_words=CLINICAL_STOP_WORDS, ngram_range=(1,2),
                           max_features=5000, sublinear_tf=True)
Xtr_t = vect_re.fit_transform(X_tr); Xte_t = vect_re.transform(X_te)

classifiers = {
    "Logistic Reg"  : LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42),
    "Complement NB" : ComplementNB(),
    "Linear SVM"    : LinearSVC(C=1,class_weight="balanced",random_state=42,max_iter=2000),
}
print("Readmission from notes — classifier comparison:\n")
re_results = []
for name, clf_re in classifiers.items():
    clf_re.fit(Xtr_t, yr_tr)
    if hasattr(clf_re, "predict_proba"):
        proba = clf_re.predict_proba(Xte_t)[:,1]
    else:
        proba = clf_re.decision_function(Xte_t)
    auc = roc_auc_score(yr_te, proba)
    f1  = f1_score(yr_te, clf_re.predict(Xte_t))
    re_results.append({"Classifier":name,"AUC":round(auc,4),"F1":round(f1,4)})
    print(f"  {name:16s}: AUC={auc:.4f} | F1={f1:.4f}")
display(pd.DataFrame(re_results).sort_values("AUC",ascending=False))


## 6. Log-Odds Discriminative Terms

In [ ]:
lr_re = LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=42)
lr_re.fit(Xtr_t, yr_tr)
fn_re  = vect_re.get_feature_names_out()
coef_r = lr_re.coef_[0]

top_pos = np.argsort(coef_r)[-15:][::-1]
top_neg = np.argsort(coef_r)[:15]

fig, axes = plt.subplots(1,2,figsize=(14,6))
for ax, indices, title, color in [
    (axes[0], top_pos, "Top terms → READMITTED",     "#D65F5F"),
    (axes[1], top_neg, "Top terms → NOT readmitted",  "#4878CF"),
]:
    terms = [fn_re[i] for i in indices]; vals = [coef_r[i] for i in indices]
    ax.barh(terms[::-1], vals[::-1], color=color, alpha=0.85, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("Log-odds coefficient"); ax.set_title(title, fontsize=10)
    for i,(t,v) in enumerate(zip(terms[::-1],vals[::-1])):
        ax.text(v+(0.005 if v>0 else -0.005),i,f"{v:.3f}",
                va="center",ha="left" if v>0 else "right",fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/mod05/log_odds_readmission.png", bbox_inches="tight"); plt.show()


## 7. Hybrid Text + Structured Features

In [ ]:
sc = StandardScaler()
X_struct_tr = csr_matrix(sc.fit_transform(df.loc[X_tr.index, ["note_length"]]))
X_struct_te  = csr_matrix(sc.transform(df.loc[X_te.index, ["note_length"]]))

X_hybrid_tr = hstack([Xtr_t, X_struct_tr])
X_hybrid_te  = hstack([Xte_t, X_struct_te])

lr_hybrid = LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42)
lr_hybrid.fit(X_hybrid_tr, yr_tr)
auc_hyb  = roc_auc_score(yr_te, lr_hybrid.predict_proba(X_hybrid_te)[:,1])
auc_text = roc_auc_score(yr_te, lr_re.predict_proba(Xte_t)[:,1])
print(f"Text-only AUC    : {auc_text:.4f}")
print(f"Text+struct AUC  : {auc_hyb:.4f}")
print(f"Improvement      : {auc_hyb-auc_text:+.4f}")


## 8. Knowledge Check
1. TF-IDF bigrams score AUC=0.73 vs unigrams 0.71. What do bigrams capture in clinical text?
2. ComplementNB outperforms Logistic Regression. When is CNB typically better for text?
3. Character n-grams perform poorly here. Why?
4. 'heart failure' as a bigram vs unigrams 'heart' and 'failure' — which is more informative?
5. Add a third feature: extract BNP and creatinine values via regex and include as numeric features.

In [ ]:
# Exercise 5 — regex numeric features
bnp_pat = r'BNP\s*(\d{2,5})'
cr_pat  = r'Cr\s*(\d\.\d)'

def get_bnp(text):
    m = re.search(bnp_pat, text, re.IGNORECASE)
    return float(m.group(1)) if m else np.nan

def get_cr(text):
    m = re.search(cr_pat, text, re.IGNORECASE)
    return float(m.group(1)) if m else np.nan

bnp_tr = df.loc[X_tr.index,"text"].apply(get_bnp).fillna(0).values.reshape(-1,1)
bnp_te  = df.loc[X_te.index,"text"].apply(get_bnp).fillna(0).values.reshape(-1,1)
cr_tr  = df.loc[X_tr.index,"text"].apply(get_cr).fillna(0).values.reshape(-1,1)
cr_te   = df.loc[X_te.index,"text"].apply(get_cr).fillna(0).values.reshape(-1,1)

X_aug_tr = hstack([Xtr_t, csr_matrix(sc.fit_transform(np.hstack([bnp_tr,cr_tr])))])
X_aug_te  = hstack([Xte_t, csr_matrix(sc.transform(np.hstack([bnp_te,cr_te])))])

lr_aug = LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42)
lr_aug.fit(X_aug_tr, yr_tr)
auc_aug = roc_auc_score(yr_te, lr_aug.predict_proba(X_aug_te)[:,1])
print(f"Text only     : {auc_text:.4f}")
print(f"Text+BNP+Cr   : {auc_aug:.4f}  (gain: {auc_aug-auc_text:+.4f})")


***
**Next → NB-04: Word Embeddings & Word2Vec**
